## Browser Proxy Routing 검증

이 Notebook은 AgentCore Browser가 `agentcore-browser-proxy.yaml`로 배포한 Squid proxy를 통해
traffic을 routing하는지 검증합니다.

Stack은 다음 항목을 배포합니다.
- Public subnet의 EC2에 **Squid proxy**(Secrets Manager를 통한 basic auth 사용)
- Private subnet에 **AgentCore Browser**(VPC mode, egress를 Squid:3128로 제한)
- 5분마다 Squid access log를 수신하는 **S3 bucket**

### 사전 요구 사항

1. `agentcore-browser-proxy.yaml` CloudFormation stack 배포
2. Dependency를 설치하고 **kernel 다시 시작**:

In [ ]:
!pip install -qU -r requirements.txt

### 1. CloudFormation output 읽기

Stack에서 Browser ID, Squid IP 및 Secrets Manager ARN을 가져옵니다.

In [ ]:
import boto3
from urllib.parse import urlparse
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

STACK_NAME = "agentcore-browser-proxy"

session = boto3.Session()
REGION = session.region_name
print(f"Region: {REGION}")
browser_client = session.client("bedrock-agentcore")

cfn = session.client("cloudformation")
outputs = {o["OutputKey"]: o["OutputValue"] for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]}

BROWSER_ID = outputs["BrowserId"]
SQUID_IP = outputs["SquidPrivateIp"]
SQUID_PUBLIC_IP = outputs["SquidPublicIp"]
SECRET_ARN = outputs["ProxySecretArn"]
LOG_BUCKET = outputs["LogBucketName"]

print(f"Browser ID:       {BROWSER_ID}")
print(f"Squid private IP: {SQUID_IP}")
print(f"Squid public IP:  {SQUID_PUBLIC_IP}")
print(f"Log bucket:       {LOG_BUCKET}")

### 2. Proxy가 적용된 브라우저 세션 시작

Squid instance를 가리키는 `proxyConfiguration`을 구성하고 세션을 시작합니다.
브라우저는 모든 web traffic을 proxy를 통해 routing합니다.

In [ ]:
proxy_config = {
    "proxies": [
        {
            "externalProxy": {
                "server": SQUID_IP,
                "port": 3128,
                "credentials": {"basicAuth": {"secretArn": SECRET_ARN}},
            }
        }
    ]
}

response = browser_client.start_browser_session(
    browserIdentifier=BROWSER_ID,
    proxyConfiguration=proxy_config,
)
session_id = response["sessionId"]
ws_url = f"wss://bedrock-agentcore.{REGION}.amazonaws.com/browser-streams/{BROWSER_ID}/sessions/{session_id}/automation"
print(f"Session ID: {session_id}")

# SigV4로 WebSocket URL 서명
credentials = session.get_credentials()
https_url = ws_url.replace("wss://", "https://")
parsed = urlparse(https_url)
request = AWSRequest(method="GET", url=https_url, headers={"host": parsed.netloc})
SigV4Auth(credentials, "bedrock-agentcore", REGION).add_auth(request)
headers = {k: v for k, v in request.headers.items()}

### 3. Proxy routing 검증

Playwright로 연결해 IP 확인 서비스로 이동합니다.
Proxy가 정상적으로 작동하면 확인된 IP가 Squid instance의 public IP와 일치해야 합니다.

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(ws_url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()

    print("Checking browser's public IP via icanhazip.com...")
    await page.goto("https://icanhazip.com", timeout=15000, wait_until="domcontentloaded")
    observed_ip = (await page.inner_text("body")).strip()

    print(f"\n{'=' * 50}")
    print(f"Expected IP (Squid public): {SQUID_PUBLIC_IP}")
    print(f"Observed IP (browser):      {observed_ip}")
    match = observed_ip == SQUID_PUBLIC_IP
    print(f"Result: {'PASS' if match else 'FAIL'} — traffic {'is' if match else 'is NOT'} routed through proxy")
    print(f"{'=' * 50}")

    await browser.close()

### 4. 세션 중지

In [ ]:
browser_client.stop_browser_session(browserIdentifier=BROWSER_ID, sessionId=session_id)
print(f"Session {session_id} stopped")

### 문제 해결

- **IP 불일치**: Browser security group이 Squid:3128로의 egress만 허용하는지 확인합니다.
- **연결 timeout**: EC2 instance에 SSH로 접속해 `systemctl status squid`를 확인하고 Squid가 실행 중인지 검증합니다.
- **Auth 오류**: Secrets Manager secret이 Squid htpasswd와 일치하는지 확인하고 instance의 `/var/log/squid/access.log`를 살펴봅니다.
- **S3 log 없음**: Log는 cron을 통해 5분마다 동기화됩니다. Instance의 `/var/log/user-data.log`에서 설정 오류를 확인합니다.